<a href="https://www.kaggle.com/code/pavankumar960/stocks-prediction-lstm-model?scriptVersionId=244246245" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Project Overview

- Working on Reliance Dataset
- GPU is on for this Notebook
- Importing Required Libraries

In [ ]:
#Basic Libs
import pandas as pd
import numpy as np

#Graph Libs
import matplotlib.pyplot as plt
import seaborn as sns

#Model Training Libs
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras import Input

import warnings 
warnings.filterwarnings('ignore')

# Data Loading

In [ ]:
df = pd.read_csv('/kaggle/input/nifty50-stock-market-data/RELIANCE.csv', parse_dates=['Date'])
df.set_index('Date', inplace=True)
df = df[['Close']]

# Data Preprocessing

In [ ]:
price_scaler = MinMaxScaler()
scaled_prices = price_scaler.fit_transform(df)

In [ ]:
def create_sequences(data, seq_length):
    X_sequences, y_targets = [], []
    for i in range(seq_length, len(data)):
        X_sequences.append(data[i-seq_length:i])
        y_targets.append(data[i])
    return np.array(X_sequences), np.array(y_targets)

sequence_length = 60
X_sequences, y_targets = create_sequences(scaled_prices, sequence_length)

# Train-Test Split

In [ ]:
split_idx = int(0.8 * len(X_sequences))
X_train, X_test = X_sequences[:split_idx], X_sequences[split_idx:]
y_train, y_test = y_targets[:split_idx], y_targets[split_idx:]

# Model - LSTM

In [ ]:
lstm_model = Sequential([
    Input(shape=(sequence_length, 1)),
    LSTM(50, return_sequences=True),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mean_squared_error')

# Model Training

In [ ]:
history = lstm_model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Price Prediction

In [ ]:
predicted_scaled_prices = lstm_model.predict(X_test)
actual_scaled_prices = y_test

predicted_prices = price_scaler.inverse_transform(predicted_scaled_prices)
actual_prices = price_scaler.inverse_transform(actual_scaled_prices)


# Results & Visualization

In [ ]:
mae = mean_absolute_error(actual_prices, predicted_prices)
r2 = r2_score(actual_prices, predicted_prices)
print("MAE:", mae, "R² Score:", r2)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual_prices, label='Actual Closing Price')
plt.plot(predicted_prices, label='Predicted Closing Price')
plt.title('RELIANCE: LSTM Predicted vs Actual Closing Price')
plt.xlabel('Days')
plt.ylabel('Price (INR)')
plt.legend()
plt.show()

# Conclusion

- Predicted the next day’s closing price using the past 60 days of closing prices.
- Used an LSTM model, suitable for time series patterns like trend, momentum, and seasonality.
- Dataset split into 80% for training and 20% for testing.